In [12]:
import arcpy
import os

In [13]:
# ── PATHS ──────────────────────────────────────────────────────────────────────────────────────────────────

# Input shorelines polygon feature class
shore_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb\CARIBBEAN_SHORELINES_THESIS_p26920"

# Input LF/HF point feature classes for GIS calculations of mission background conditions
lf_points_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_LF_background_points_p26920"
hf_points_p26920 = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_HF_background_points_p26920"

In [14]:
# ── SETTINGS ───────────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True

In [15]:
############### NOTE ###############
# 'Near' tool edits Attribute Table of input feature class in-place (not additional output files/etc.)
# Edits are saved to source feature class in .gdb (not just to layer on map)
####################################

In [16]:
# ── 1. SHORELINE CALCULATIONS ──────────────────────────────────────────────────────────────────────────────

# Near (Analysis Tools): Function
def run_near_to_shore(in_points, shore_features):
    arcpy.analysis.Near(
        in_features=in_points,
        near_features=shore_features,
        search_radius=None,
        # write x- and y-coordinates of closest shoreline to NEAR_X and NEAR_Y fields
        location="LOCATION",
        # do not calculate angle of line from point to nearest shoreline (not useful in terms of my thesis)
        angle="NO_ANGLE",
        # * geodesic uses input features' GEOGRAPHIC coordinate system (EPSG:4269) even if projected coordinate system is present = NEAR_X/NEAR_Y coordinates reported in decimal degrees + NEAR_DIST computed as shortest path on a spheroid (accounts for curvature of earth) between point and shoreline polygon
        method="GEODESIC",
        # report NEAR_DIST values in meters
        distance_unit="Meters"
    )
    print(f"✅ Near tool complete for {os.path.basename(in_points)}")

# call function for LF and HF mission background conditions points
run_near_to_shore(lf_points_p26920, shore_p26920)
run_near_to_shore(hf_points_p26920, shore_p26920)

✅ Near tool complete for calc_LF_background_points_p26920
✅ Near tool complete for calc_HF_background_points_p26920


In [17]:
# Fields added to Attribute Table by 'Near' tool: 
# NEAR_FID (FID of closest shoreline feature), NEAR_DIST (distance to closest feature in meters (geodesic)), NEAR_X (x-coordinate of closest shoreline), NEAR_Y (y-coordinate of closest shoreline)

In [18]:
# ── RENAME FIELDS ADDED BY NEAR TOOL  ──────────────────────────────────────────────────────────────────────

# Map of 'Near' tool output field names to new desired field names
rename_map = {
    "NEAR_DIST" :   "distance_to_shore_m",
    "NEAR_FID"  :   "nearest_shoreline_fid",
    "NEAR_X"    :   "nearest_shoreline_x_coordinate_DD",
    "NEAR_Y"    :   "nearest_shoreline_y_coordinate_DD",
}

# Apply renaming map to each point layer
for table in (lf_points_p26920, hf_points_p26920):
    # get list of field names currently in Attribute Table
    existing = {f.name for f in arcpy.ListFields(table)}
    # for loop to rename fields
    for old_name, new_name in rename_map.items():
        # print warning message if original 'Near' tool output field name is NOT present
        if old_name not in existing:
            print(f"\n⚠️[WARNING] Field '{old_name}' not found in {os.path.basename(table)} - skipped \n")
            continue
        # print warning message if a field with the desired new name already exists
        if new_name in existing:
            print(f"\n⚠️[WARNING] Field '{new_name}' already exists in {os.path.basename(table)} - '{old_name}' not renamed \n")
            continue
        # attempt fields based on 'rename_map' defined above
        try:
            # 'Alter Field' tool renames field (and its alias) WITHOUT losing data
            arcpy.management.AlterField(table, old_name, new_name, new_name)
        # print warning message if renaming fails for any fields
        except Exception as e:
            print(f"\n❌[FAILURE] Failed to rename '{old_name}' to '{new_name}' in {os.path.basename(table)}: {e} \n")
    print(f"✅ Fields renamed for {os.path.basename(table)}")

✅ Fields renamed for calc_LF_background_points_p26920
✅ Fields renamed for calc_HF_background_points_p26920


In [19]:
# ── 2. ADD NEAREST SHORELINE NAMES (COUNTRY, SOVEREIGN, ISLAND) TO POINT ATRRIBUTE TABLES ───────────────────

# Build lookup dictionary {} to match "OBJECTID" to "COUNTRY, SOVEREIGN, ISLAND" (non-NA cells only)
shoreline_names = {}
# read objectid field + three name fields (country, sovereign, island) from shorelines table
with arcpy.da.SearchCursor(shore_p26920, ["OBJECTID", "COUNTRY", "SOVEREIGN", "ISLAND"]) as cursor:
    # loop through the four key fields in each row of shorelines table
    for oid, country, sovereign, island in cursor:
        # skip cells with NA or blank strings
        parts = [p for p in (country, sovereign, island) if p not in (None, "", " ")]
        # join any of the three name fields containing values (non-NA) with commas; output empty string if all three name fields are blank
        shoreline_names[oid] = ", ".join(parts)

# Function to add + populate "nearest_shoreline_name" field in mission background conditions point feature class Attribute Table
def populate_shoreline_name(in_points, name_lookup):
    # add new "nearest_shoreline_name" field
    arcpy.management.AddField(in_points, "nearest_shoreline_name", "TEXT", field_length=255)
    # populate rows in "nearest_shoreline_name" field by matching "nearest_shoreline_fid" values in mission points Attribute Table to "oid" key values in shoreline_names{} dictionary
    with arcpy.da.UpdateCursor(in_points, ["nearest_shoreline_fid", "nearest_shoreline_name"]) as cursor:
        # loop through each row in mission points Attribute Table
        for row in cursor:
            # look up combined name (country, sovereign, island) for matching "nearest_shoreline_fid" (output empty string if not found)
            row[1] = name_lookup.get(row[0], "")
            # write value back to row
            cursor.updateRow(row)
    print(f"✅ 'nearest_shoreline_name' field created and populated for {os.path.basename(in_points)}")

# call function for LF and HF mission background conditions points
populate_shoreline_name(lf_points_p26920, shoreline_names)
populate_shoreline_name(hf_points_p26920, shoreline_names)

✅ 'nearest_shoreline_name' field created and populated for calc_LF_background_points_p26920
✅ 'nearest_shoreline_name' field created and populated for calc_HF_background_points_p26920


In [ ]:
# ── STANDARDIZE EMPTY CELL VALUES TO NULL + WARN IF ROWS MISSING NEAR TOOL DATA ────────────────────────────

# fields to clean and check on each point layer
check_fields = [
    "nearest_shoreline_fid",
    "distance_to_shore_m",
    "nearest_shoreline_x_coordinate_DD",
    "nearest_shoreline_y_coordinate_DD",
    "nearest_shoreline_name",
]

# fields that indicate genuine Near-tool failure if Null ("nearest_shoreline_name" excluded - Null cells for this field are valid and simply due to matched shoreline row missing country/sovereign/island)
critical_fields = [
    "nearest_shoreline_fid",
    "distance_to_shore_m",
    "nearest_shoreline_x_coordinate_DD",
    "nearest_shoreline_y_coordinate_DD",
]

# Loop through both point feature classes (lf_points_p26920, hf_points_p26920) and examine Attribute Table contents
for table in (lf_points_p26920, hf_points_p26920):
    # get field names present in point layer's Attribute Table
    existing = {f.name for f in arcpy.ListFields(table)}
    # find any check_fields missing in table
    absent = [f for f in check_fields if f not in existing]
    # print warning message if any expected fields are missing and skip table
    if absent:
        print(f"\n⚠️[WARNING] {os.path.basename(table)} missing expected field(s): {', '.join(absent)} -- skipping \n")
        continue

    # if all check_fields were successfully found in table, standardize missing data cells to true Null
    with arcpy.da.UpdateCursor(table, check_fields) as cursor:
        # loop through each row
        for row in cursor:
            # track whether this row was modified
            changed = False
            # inspect each field value in the row
            for i, value in enumerate(row):
                # replace -1 ('Near' tool output for cells with NO nearest shoreline found) with true Null
                if value == -1:
                    row[i] = None
                    changed = True
                # replace empty/blank strings (no country/sovereign/island in matching row of shorelines table) with true Null
                elif isinstance(value, str) and value.strip() == "":
                    row[i] = None
                    changed = True
            # write fixed value back to row only if something was changed
            if changed:
                cursor.updateRow(row)

    # after cleaning, count rows that contain Null cells in any of the "critical_fields" fields (defined above)
    missing_count = 0
    # read critical fields in each row
    with arcpy.da.SearchCursor(table, critical_fields) as cursor:
        # loop through each row
        for row in cursor:
            # flag row if any checked field is Null
            if any(value is None for value in row):
                # increment missing counter
                missing_count += 1
    # warn if any missing values remain, otherwise confirm all populated
    if missing_count > 0:
        print(f"\n⚠️[WARNING] {missing_count} row(s) in {os.path.basename(table)} have missing (Null) values in checked fields \n")
    else:
        print(f"✅ All checked fields populated for {os.path.basename(table)}")

✅ All checked fields populated for calc_LF_background_points_p26920
✅ All checked fields populated for calc_HF_background_points_p26920
